# VAZHI SFT v7.1 — Gemma 3 1B-it (Incremental, LoRA r=16)

**Incremental training on top of v7.0** with doubled LoRA rank to test whether
Gemma 3 1B-it can tolerate more aggressive training without catastrophic forgetting.

```
Lineage: google/gemma-3-1b-it → SFT v7.0 (r=8) → SFT v7.1 (r=16, this notebook)
```

**v7.0 result:** Conservative LoRA (r=8, q_proj+v_proj) preserved Tamil (94% word)
but was **too weak to override pretrained priors** — identity still says "Google LLM",
factual corrections not taken (capital still Coimbatore instead of Chennai).
Partial success: "நான் VAZHI" appeared in greeting.

**v7.1 strategy:** Incremental — stack r=16 LoRA on the v7.0 merged model.
The model already has partial SFT learning (task behavior, partial identity).
Higher rank gives more capacity to push identity and factual corrections through.

**Changes from v7.0:**
- Base model: `google/gemma-3-1b-it` → **`CryptoYogi/vazhi-v7_0`** (incremental)
- LoRA r=8 → **r=16** (2x more trainable params)
- LoRA alpha=16 → **alpha=32** (maintain 2x ratio)
- Cell 6 split bug fixed (checks "test" in addition to "validation")
- All else identical: same dataset, LR, targets, 1 epoch

**Success criteria:**
1. Tamil word score stays >= 84% (max 10% drop from 94% baseline)
2. Identity learned ("நான் VAZHI" not "Google LLM")
3. Factual corrections taken (Chennai, not Coimbatore)
4. No repetition/degeneration

**Dataset:** `CryptoYogi/vazhi-tamil-sft-v7_0` (4,172 samples: 3,754 train + 418 eval)
- Spiritual 39.2%, domain 51.8%, identity 5.5%, safety 0.8%
- 61 mission pairs, 29 correction pairs, avg 47 words

**Runtime:** Colab Pro GPU (L4 recommended, A100 ideal)

In [1]:
# Cell 1 — Dependencies
!pip install -q -U \
  "transformers>=4.50.0,<5.0.0" \
  "trl>=0.20.0" \
  "datasets>=2.21.0" \
  "peft>=0.13.0" \
  "accelerate>=0.34.0" \
  "huggingface_hub>=0.24.7"

import torch
print(f"\u2705 Dependencies installed")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"   VRAM: {vram / 1024**3:.0f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 144.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.2 MB/s eta 0:00:00
✅ Dependencies installed
   PyTorch: 2.9.0+cu128
   CUDA: True
   GPU: NVIDIA A100-SXM4-80GB
   VRAM: 79 GB


In [2]:
# Cell 2 — Configuration

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import re
import random
import gc
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset, Dataset
from huggingface_hub import login, HfApi

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === ENVIRONMENT DETECTION ===
IS_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if IS_KAGGLE else "/content"
ENV_NAME = "Kaggle" if IS_KAGGLE else "Colab"

# === KEY CONFIG ===
BASE_MODEL = "CryptoYogi/vazhi-v7_0"               # Incremental: v7.0 merged model
VANILLA_MODEL = "google/gemma-3-1b-it"              # For reference/baseline comparison
SFT_DATASET = "CryptoYogi/vazhi-tamil-sft-v7_0"    # Same dataset as v7.0
OUTPUT_MODEL = "CryptoYogi/vazhi-v7_1"              # v7.1: incremental + LoRA r=16
ADAPTER_REPO = "CryptoYogi/vazhi-v7_1-lora"         # Adapter backup

# Training config — v7.1: Incremental on v7.0 with doubled LoRA rank
# v7.0 (r=8 on vanilla) preserved Tamil but couldn't override identity/facts.
# v7.1 stacks r=16 on top of v7.0 — effectively epoch 2 with more capacity.
LEARNING_RATE = 1e-5       # Same as v7.0 (proven safe)
NUM_EPOCHS = 1             # 1 more epoch on top of v7.0's 1 epoch
MAX_LENGTH = 2048          # Same as v7.0
LORA_R = 16                # v7.0 was 8 — doubled to push identity/factual corrections
LORA_ALPHA = 32            # Maintain 2x ratio (v7.0 was 16)
LORA_TARGETS = ["q_proj", "v_proj"]  # Same targets as v7.0
BATCH_SIZE = 4             # Per-device (adjust: 2=T4, 4=L4, 8=A100)
GRADIENT_ACCUMULATION = 4  # Effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION

# Gemma 3 does NOT officially support system role in its chat template.
# System instructions are embedded in the first user message instead.
SYSTEM_PROMPT = (
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0bb5\u0bb4\u0bbf (VAZHI), "
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bc1 \u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bc1\u0b95\u0bcd\u0b95\u0bbe\u0ba9 "
    "AI \u0b89\u0ba4\u0bb5\u0bbf\u0baf\u0bbe\u0bb3\u0bb0\u0bcd. "
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0ba4\u0bae\u0bbf\u0bb4\u0bbf\u0bb2\u0bcd \u0baa\u0ba4\u0bbf\u0bb2\u0bb3\u0bbf\u0baa\u0bcd\u0baa\u0bc0\u0bb0\u0bcd\u0b95\u0bb3\u0bcd."
)

# GPU auto-detection
assert torch.cuda.is_available(), "GPU required! Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0).lower()
_props = torch.cuda.get_device_properties(0)
VRAM_GB = getattr(_props, 'total_memory', getattr(_props, 'total_mem', 0)) / 1e9
IS_HIGH_END_GPU = any(x in gpu_name for x in ["a100", "l4", "h100", "a10"])
USE_BF16 = IS_HIGH_END_GPU
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
n_gpus = torch.cuda.device_count()
effective_batch = BATCH_SIZE * n_gpus * GRADIENT_ACCUMULATION

print(f"\u2705 SFT v7.1 Configuration (Gemma 3 — Incremental + LoRA r=16):")
print(f"   Environment: {ENV_NAME}")
print(f"   Base model:  {BASE_MODEL} (v7.0 merged — incremental)")
print(f"   Vanilla ref: {VANILLA_MODEL}")
print(f"   Dataset:     {SFT_DATASET} (same as v7.0)")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"   LR:          {LEARNING_RATE}")
print(f"   LoRA:        r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGETS}")
print(f"   v7.0 was:    r=8, alpha=16 on vanilla google/gemma-3-1b-it")
print(f"   Batch:       {BATCH_SIZE} x {GRADIENT_ACCUMULATION} accum = {effective_batch} effective")
print(f"   GPU:         {torch.cuda.get_device_name(0)} ({VRAM_GB:.0f} GB)")
print(f"   Precision:   {'bf16' if USE_BF16 else 'fp16'}")

✅ SFT v7.1 Configuration (Gemma 3 — Incremental + LoRA r=16):
   Environment: Colab
   Base model:  CryptoYogi/vazhi-v7_0 (v7.0 merged — incremental)
   Vanilla ref: google/gemma-3-1b-it
   Dataset:     CryptoYogi/vazhi-tamil-sft-v7_0 (same as v7.0)
   Output:      CryptoYogi/vazhi-v7_1
   LR:          1e-05
   LoRA:        r=16, alpha=32, targets=['q_proj', 'v_proj']
   v7.0 was:    r=8, alpha=16 on vanilla google/gemma-3-1b-it
   Batch:       4 x 4 accum = 16 effective
   GPU:         NVIDIA A100-SXM4-80GB (85 GB)
   Precision:   bf16


In [3]:
# Cell 3 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("\u2705 Logged in to HuggingFace")

✅ Logged in to HuggingFace


In [4]:
# Cell 4 — Load Tokenizer + Helper Functions
#
# Use vanilla Gemma 3 tokenizer (same tokenizer for v7.0 and vanilla).
# v7.0 model uses the same tokenizer — no special tokens were added.

tokenizer = AutoTokenizer.from_pretrained(VANILLA_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\u2705 Tokenizer: {len(tokenizer)} tokens (from {VANILLA_MODEL})")
print(f"   pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"   eos_token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# Verify chat template works
test_msgs = [{"role": "user", "content": "test"}]
test_formatted = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print(f"   Chat template test: {repr(test_formatted[:80])}")

# Check if system role is supported in the template
SYSTEM_ROLE_SUPPORTED = True
try:
    test_sys = [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "test"},
    ]
    sys_formatted = tokenizer.apply_chat_template(test_sys, tokenize=False, add_generation_prompt=True)
    print(f"   System role: supported")
    print(f"   System test: {repr(sys_formatted[:120])}")
except Exception as e:
    SYSTEM_ROLE_SUPPORTED = False
    print(f"   System role: NOT supported ({e})")
    print(f"   Will embed system prompt in first user message")


def build_messages(user_text, system_text=None):
    """Build message list for Gemma 3 chat template."""
    msgs = []
    if system_text and SYSTEM_ROLE_SUPPORTED:
        msgs.append({"role": "system", "content": system_text})
        msgs.append({"role": "user", "content": user_text})
    elif system_text:
        # Embed system prompt in first user message
        msgs.append({"role": "user", "content": f"{system_text}\n\n{user_text}"})
    else:
        msgs.append({"role": "user", "content": user_text})
    return msgs


def build_chat_prompt(user_text):
    """Build a formatted prompt string for generation."""
    msgs = build_messages(user_text, SYSTEM_PROMPT)
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True,
    )


def tamil_char_pct(text):
    if not text:
        return 0.0
    total = sum(1 for c in text if not c.isspace() and not c.isdigit())
    if total == 0:
        return 0.0
    tamil = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    return 100.0 * tamil / total


def tamil_word_score(text):
    """Score based on Tamil word validation (not just char %)."""
    words = text.split()
    if not words:
        return 0.0, 0, 0
    tamil_words = 0
    for w in words:
        clean = re.sub(r'[\d\W]', '', w)
        if not clean:
            continue
        tamil_chars = sum(1 for c in clean if '\u0B80' <= c <= '\u0BFF')
        if tamil_chars / len(clean) > 0.5:
            tamil_words += 1
    return 100.0 * tamil_words / len(words), tamil_words, len(words)


def compute_repeat_ratio(text, n=3):
    """Detect repetitive output via n-gram ratio."""
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    return 1.0 - len(set(ngrams)) / len(ngrams)


def extract_response(full_text):
    """Extract model response from Gemma formatted output."""
    # Gemma format: <start_of_turn>model\n{response}<end_of_turn>
    if "<start_of_turn>model" in full_text:
        # Get the last model turn
        parts = full_text.split("<start_of_turn>model")
        resp = parts[-1]
        if "<end_of_turn>" in resp:
            resp = resp.split("<end_of_turn>")[0]
        resp = resp.strip()
        if resp.startswith("\n"):
            resp = resp[1:]
        return resp.strip()
    return full_text.strip()


def generate_response(model, prompt_text, max_new_tokens=200):
    """Generate a response using Gemma 3 chat format."""
    full_prompt = build_chat_prompt(prompt_text)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only new tokens
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=False)

    # Clean up end-of-turn markers
    if "<end_of_turn>" in response:
        response = response.split("<end_of_turn>")[0]
    # Remove any remaining special tokens
    response = response.replace("<eos>", "").replace("<bos>", "").strip()
    return response


print("\u2705 Helpers ready")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ Tokenizer: 262145 tokens (from google/gemma-3-1b-it)
   pad_token: <pad> (id=0)
   eos_token: <eos> (id=1)
   Chat template test: '<bos><start_of_turn>user\ntest<end_of_turn>\n<start_of_turn>model\n'
   System role: supported
   System test: '<bos><start_of_turn>user\nYou are helpful.\n\ntest<end_of_turn>\n<start_of_turn>model\n'
✅ Helpers ready


In [5]:
# Cell 5 — Pre-SFT Baseline on v7.0 Model
#
# Record v7.0 model outputs BEFORE v7.1 training for comparison.
# v7.0 baseline: Tamil 92% char / 94% word, identity partially learned.
# We need to see if v7.1 (r=16 incremental) improves identity/factual
# without degrading Tamil quality.

print(f"\U0001f4ca Pre-SFT Baseline: {BASE_MODEL} (v7.0)")
print("=" * 60)

BASELINE_PROMPTS = [
    ("\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "greeting"),
    ("\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "identity"),
    ("\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "thanks"),
    ("\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "factual"),
    ("\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "govt"),
    ("\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "govt"),
    ("\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "health"),
    ("\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1", "safety"),
    ("\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "culture"),
    ("\u0b95\u0bbe\u0baf\u0bcd\u0b9a\u0bcd\u0b9a\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "health"),
]

print(f"\n\U0001f4e5 Loading {BASE_MODEL} for baseline...")
baseline_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=MODEL_DTYPE, device_map={"":0},
)
baseline_model.eval()
baseline_model.config.use_cache = True

pre_sft_results = []
for prompt_text, category in BASELINE_PROMPTS:
    resp = generate_response(baseline_model, prompt_text)
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    pre_sft_results.append({
        'prompt': prompt_text, 'category': category,
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
    })
    print(f"\n[{category}] Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")

avg_pre_char = np.mean([r['tamil_char_pct'] for r in pre_sft_results])
avg_pre_word = np.mean([r['tamil_word_pct'] for r in pre_sft_results])
avg_pre_rep = np.mean([r['repeat_ratio'] for r in pre_sft_results])
print(f"\n\U0001f4ca v7.0 baseline: char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%, rep {avg_pre_rep:.2f}")
print(f"   (v7.0 eval was: char 92%, word 94%)")
print(f"   (vanilla was:   char 93%, word 95%)")

del baseline_model
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Baseline model freed")

📊 Pre-SFT Baseline: CryptoYogi/vazhi-v7_0 (v7.0)

📥 Loading CryptoYogi/vazhi-v7_0 for baseline...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]


[greeting] Char: 56%, Word: 46%, Rep: 0.00
  Q: வணக்கம்
  A: வணக்கம்! நான் உங்களுக்கு எப்படி உதவ முடியும்? 

(Hello!) How can I help you today?

[identity] Char: 97%, Word: 100%, Rep: 0.00
  Q: நீங்கள் யார்?
  A: நான் ஒரு பெரிய மொழி மாதிரி, கூகிளால் பயிற்சி அளிக்கப்பட்டது. நான் உங்களுக்காக இருக்கிறேன்! உங்கள் கேள்விகளுக்குப் பதில் அளிக்கவும் அல்லது உரையாடலைத் தொடரவும் தயங்க வேண்டாம். 

உங்களுக்கு என்ன தேவை?

[thanks] Char: 92%, Word: 94%, Rep: 0.00
  Q: நன்றி
  A: சரியாக! நான் உங்களுக்கு எப்படி உதவ முடியும்? என்னிடம் கேட்க விரும்பும் கேள்விகள் அல்லது உதவி தேவைப்பட்டால் தயங்காமல் கேளுங்கள். 

(Vazhi)

[factual] Char: 97%, Word: 100%, Rep: 0.00
  Q: தமிழ்நாட்டின் தலைநகரம் எது?
  A: தமிழ்நாட்டின் தற்போதைய தலைநகரம் கோயம்புத்தூர் ஆகும். 

இது ஒரு துல்லியமான பதில் என்று நம்புகிறேன்! வேறு ஏதாவது உங்களுக்கு உதவ வேண்டுமா?

[govt] Char: 85%, Word: 90%, Rep: 0.00
  Q: ரேஷன் கார்டு பற்றி தகவல் தேவை
  A: சாரி, நான் ஒரு பெரிய மொழி மாதிரி மட்டுமே. எனக்கு ரேஷன் கார்டுகளின் தகவல்களைத் தேட முடியாது. ஆன

In [6]:
# Cell 6 — Load & Format Dataset v7.0 (instruction/output → Gemma format)
#
# v7.0 dataset has raw instruction/output columns (NOT ChatML).
# We apply Gemma 3's chat template here at training time.
#
# FIX from v7.0: HuggingFace auto-maps "-eval.json" to "test" split,
# not "validation". Now checks both to avoid accidental re-split.

print(f"\U0001f4da Loading SFT dataset from {SFT_DATASET}...")

try:
    sft_ds = load_dataset(SFT_DATASET)
    if "train" in sft_ds and ("validation" in sft_ds or "test" in sft_ds):
        raw_train = sft_ds["train"]
        raw_eval = sft_ds.get("validation", sft_ds.get("test"))
    elif "train" in sft_ds:
        raw_train = sft_ds["train"]
        raw_eval = None
    else:
        raise KeyError("No train split found")
except (KeyError, ValueError, Exception) as e:
    print(f"   HF load failed ({e}), trying JSON files...")
    raw_train = load_dataset("json", data_files={
        "train": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v7_0-train.json"
    })["train"]
    raw_eval = load_dataset("json", data_files={
        "eval": f"hf://datasets/{SFT_DATASET}/vazhi-tamil-sft-v7_0-eval.json"
    })["eval"]

print(f"   Raw train: {len(raw_train)}")
print(f"   Raw eval:  {len(raw_eval) if raw_eval else 'None'}")
print(f"   Columns: {raw_train.column_names}")
print(f"   Available splits: {list(sft_ds.keys()) if 'sft_ds' in dir() else 'N/A'}")

# === Apply Gemma 3 chat template ===
def format_for_gemma(sample):
    """Convert instruction/output pair to Gemma chat format."""
    instruction = sample["instruction"]
    output = sample["output"]

    if not instruction or not output:
        return None

    # Build messages with system prompt embedded in user turn
    messages = build_messages(instruction, SYSTEM_PROMPT)

    # Add model response (Gemma uses "model" not "assistant")
    messages.append({"role": "model", "content": output})

    # Format using Gemma tokenizer's chat template
    try:
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
    except Exception:
        # Fallback: try with "assistant" role if "model" fails
        messages[-1]["role"] = "assistant"
        try:
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )
        except Exception as e2:
            print(f"   Template error: {e2}")
            return None

    return {
        "text": formatted,
        "bucket": sample.get("bucket", "unknown"),
    }


def convert_dataset(raw_ds, split_name):
    """Convert all samples in a dataset split."""
    converted = []
    failed = 0
    for i in range(len(raw_ds)):
        result = format_for_gemma(raw_ds[i])
        if result:
            converted.append(result)
        else:
            failed += 1
    print(f"   {split_name}: {len(raw_ds)} → {len(converted)} converted, {failed} failed")
    return Dataset.from_list(converted)


print("\n\U0001f500 Applying Gemma 3 chat template...")
train_ds = convert_dataset(raw_train, "train")
if raw_eval:
    eval_ds = convert_dataset(raw_eval, "eval")
else:
    # If no eval split, create from train (10%)
    full_ds = convert_dataset(raw_train, "full")
    split = full_ds.train_test_split(test_size=0.1, seed=RANDOM_SEED)
    train_ds = split["train"]
    eval_ds = split["test"]
    print(f"   Split: {len(train_ds)} train / {len(eval_ds)} eval")

print(f"\n\u2705 Formatted dataset:")
print(f"   Train: {len(train_ds)} samples")
print(f"   Eval:  {len(eval_ds)} samples")

# Spot-check: show first converted sample
print(f"\n\U0001f50d Sample (first 500 chars):")
print(train_ds[0]["text"][:500])

# Verify Gemma format tokens are present
sample_text = train_ds[0]["text"]
has_start = "<start_of_turn>" in sample_text
has_end = "<end_of_turn>" in sample_text
has_model = "<start_of_turn>model" in sample_text
has_user = "<start_of_turn>user" in sample_text
print(f"\n   Format check: start_of_turn={has_start}, end_of_turn={has_end}, model={has_model}, user={has_user}")
assert has_start and has_end and has_model and has_user, "Gemma format conversion failed!"
print("\u2705 Gemma format verified")

# Composition stats
if 'bucket' in train_ds.column_names:
    bucket_dist = Counter(train_ds['bucket'])
    total_count = sum(bucket_dist.values())
    print(f"\n\U0001f4ca Composition (train):")
    for bucket, count in sorted(bucket_dist.items(), key=lambda x: -x[1]):
        print(f"   {bucket}: {count} ({100*count/total_count:.1f}%)")

📚 Loading SFT dataset from CryptoYogi/vazhi-tamil-sft-v7_0...


README.md: 0.00B [00:00, ?B/s]

vazhi-tamil-sft-v7_0-train.json: 0.00B [00:00, ?B/s]

vazhi-tamil-sft-v7_0-eval.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3754 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/418 [00:00<?, ? examples/s]

   Raw train: 3754
   Raw eval:  418
   Columns: ['instruction', 'output', 'bucket', 'source', 'category']
   Available splits: ['train', 'test']

🔀 Applying Gemma 3 chat template...
   train: 3754 → 3754 converted, 0 failed
   eval: 418 → 418 converted, 0 failed

✅ Formatted dataset:
   Train: 3754 samples
   Eval:  418 samples

🔍 Sample (first 500 chars):
<bos><start_of_turn>user
நீங்கள் வழி (VAZHI), தமிழ்நாட்டு மக்களுக்கான AI உதவியாளர். நீங்கள் தமிழில் பதிலளிப்பீர்கள்.

Tamil Nadu govt vellinaadu padippuku scholarship tharugiradha?<end_of_turn>
<start_of_turn>model
உயர் கல்விக்கு TNEA counselling மூலம் engineering, NEET மூலம் medical, CLAT மூலம் law சேர்க்கை நடைபெறும். அரசு கல்லூரிகளில் கட்டணம் குறைவு. BC/MBC/SC/ST மாணவர்களுக்கு கட்டண விலக்கு, உதவித்தொகை உண்டு. tnteu.ac.in, annauniv.edu-ல் விவரங்கள் கிடைக்கும்.<end_of_turn>


   Format check: start_of_turn=True, end_of_turn=True, model=True, user=True
✅ Gemma format verified

📊 Composition (train):
   vazhi_packs: 2657 (70.8%)
   sa

In [7]:
# Cell 7 — Load Model + LoRA Setup
#
# v7.1: Load v7.0 merged model (incremental) and apply LoRA r=16.
# v7.0 already has 1 epoch of SFT with r=8 — we stack r=16 on top.
# This gives the model more capacity to learn identity/factual corrections
# that r=8 was too conservative to override.

print(f"\U0001f4e5 Loading {BASE_MODEL} (v7.0 merged — incremental)...")

# NO device_map for training — use .to("cuda:0") instead
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=MODEL_DTYPE,
)
model = model.to("cuda:0")

if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.gradient_checkpointing_enable()

has_device_map = hasattr(model, "hf_device_map")
print(f"   hf_device_map present: {has_device_map} (must be False)")

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 Model loaded: {model.num_parameters():,} params | GPU: {mem_gb:.1f} GB")

# Verify LoRA target modules exist in the model
print(f"\n\U0001f50d Checking LoRA target modules...")
found_modules = set()
for name, _ in model.named_modules():
    for target in LORA_TARGETS:
        if target in name:
            found_modules.add(target)
for target in LORA_TARGETS:
    status = "\u2705" if target in found_modules else "\u274c MISSING"
    print(f"   {target}: {status}")
assert len(found_modules) == len(LORA_TARGETS), f"Missing LoRA targets: {set(LORA_TARGETS) - found_modules}"

# Apply LoRA — r=16 on top of v7.0 (which used r=8 on vanilla)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

mem_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"\u2705 LoRA r={LORA_R} applied on v7.0 base | GPU: {mem_gb:.1f} GB")

📥 Loading CryptoYogi/vazhi-v7_0 (v7.0 merged — incremental)...
   hf_device_map present: False (must be False)
✅ Model loaded: 999,885,952 params | GPU: 1.9 GB

🔍 Checking LoRA target modules...
   q_proj: ✅
   v_proj: ✅
trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489
✅ LoRA r=16 applied on v7.0 base | GPU: 1.9 GB


In [8]:
# Cell 8 — Dataset Preflight: Token Length Validation

print(f"\U0001f4ca Token length distribution (train):")
token_lengths = []
for idx in range(len(train_ds)):
    tokens = tokenizer.encode(train_ds[idx]["text"], add_special_tokens=False)
    token_lengths.append(len(tokens))

token_lengths = np.array(token_lengths)
truncated = (token_lengths > MAX_LENGTH).sum()

print(f"   Mean: {token_lengths.mean():.0f}, Max: {token_lengths.max()}, P95: {np.percentile(token_lengths, 95):.0f}")
print(f"   Truncated (>{MAX_LENGTH}): {truncated} ({100*truncated/len(train_ds):.1f}%)")

# Gemma 3 uses 262K vocab — tokenization may differ from Qwen3 (151K)
# Compare average tokens per sample
avg_tokens = token_lengths.mean()
print(f"   Avg tokens/sample: {avg_tokens:.0f}")
print(f"   Total tokens: {token_lengths.sum():,}")

if truncated > len(train_ds) * 0.2:
    print(f"   \u26a0\ufe0f >20% truncated — consider increasing MAX_LENGTH or filtering long samples")
elif truncated > len(train_ds) * 0.1:
    print(f"   \u26a0\ufe0f >10% truncated — acceptable but some Sadhguru articles will be cut")
else:
    print(f"   \u2705 Token lengths OK")

📊 Token length distribution (train):
   Mean: 168, Max: 1200, P95: 577
   Truncated (>2048): 0 (0.0%)
   Avg tokens/sample: 168
   Total tokens: 630,904
   ✅ Token lengths OK


In [9]:
# Cell 9 — Training Setup

OUTPUT_DIR = f"{WORK_DIR}/sft-v7_1"

steps_per_epoch = len(train_ds) // effective_batch
total_steps = steps_per_epoch * NUM_EPOCHS
log_steps = max(total_steps // 30, 5)
eval_steps = max(steps_per_epoch // 3, 10)
save_steps = max(steps_per_epoch, 20)

print(f"\U0001f4ca Training Plan (v7.1 — LoRA r={LORA_R}):")
print(f"   Train samples:    {len(train_ds)}")
print(f"   Eval samples:     {len(eval_ds)}")
print(f"   Effective batch:  {effective_batch}")
print(f"   Steps/epoch:      ~{steps_per_epoch}")
print(f"   Total steps:      ~{total_steps}")
print(f"   Eval every:       {eval_steps} steps")
print(f"   LoRA:             r={LORA_R} (v7.0 was r=8), targets={LORA_TARGETS}")


class LossLoggingCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if "loss" in logs:
                step = state.global_step
                loss = logs["loss"]
                lr = logs.get("learning_rate", 0)
                self.losses.append((step, loss))
                print(f"  Step {step:4d}/{total_steps} | Loss: {loss:.4f} | LR: {lr:.2e}")
            if "eval_loss" in logs:
                self.eval_losses.append((state.global_step, logs["eval_loss"]))
                print(f"  \U0001f4ca Eval Loss: {logs['eval_loss']:.4f}")


class MidTrainingGenCheck(TrainerCallback):
    """Generate Tamil responses mid-training to catch catastrophic forgetting early."""

    SANITY_PROMPTS = [
        {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "label": "greeting"},
        {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "label": "identity"},
        {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "label": "factual"},
    ]

    def __init__(self, model_ref):
        self.model_ref = model_ref
        self.check_interval = max(steps_per_epoch // 2, 20)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.check_interval != 0 or state.global_step == 0:
            return
        print(f"\n  \U0001f50d Mid-training gen check (step {state.global_step}):")
        self.model_ref.eval()
        self.model_ref.config.use_cache = True
        if hasattr(self.model_ref, 'gradient_checkpointing_disable'):
            self.model_ref.gradient_checkpointing_disable()

        for item in self.SANITY_PROMPTS:
            try:
                resp = generate_response(self.model_ref, item['prompt'], max_new_tokens=80)
                tw_pct, _, _ = tamil_word_score(resp)
                rep = compute_repeat_ratio(resp)
                print(f"    [{item['label']}] Word: {tw_pct:.0f}%, Rep: {rep:.2f} | {resp[:100]}")
            except Exception as e:
                print(f"    [{item['label']}] ERROR: {e}")

        self.model_ref.train()
        self.model_ref.config.use_cache = False
        if hasattr(self.model_ref, 'gradient_checkpointing_enable'):
            self.model_ref.gradient_checkpointing_enable()


loss_cb = LossLoggingCallback()
gen_cb = MidTrainingGenCheck(model)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=log_steps,
    save_steps=save_steps,
    eval_steps=eval_steps,
    eval_strategy="steps",
    save_total_limit=3,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=1.0,
    optim="adamw_torch",
    report_to="none",
    seed=RANDOM_SEED,
    load_best_model_at_end=False,
    dataloader_pin_memory=True,
    max_length=MAX_LENGTH,
    packing=False,
    push_to_hub=True,
    hub_model_id=ADAPTER_REPO,
    hub_strategy="every_save",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    processing_class=tokenizer,
    callbacks=[loss_cb, gen_cb],
)

print(f"\u2705 SFTTrainer ready")
print(f"   Base: {BASE_MODEL} (Gemma 3 1B-it, native Tamil)")
print(f"   LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}")
print(f"   LoRA: r={LORA_R} (v7.0 was r=8), alpha={LORA_ALPHA}, targets={LORA_TARGETS}")

📊 Training Plan (v7.1 — LoRA r=16):
   Train samples:    3754
   Eval samples:     418
   Effective batch:  16
   Steps/epoch:      ~234
   Total steps:      ~234
   Eval every:       78 steps
   LoRA:             r=16 (v7.0 was r=8), targets=['q_proj', 'v_proj']


Adding EOS to train dataset:   0%|          | 0/3754 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3754 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3754 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/418 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/418 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/418 [00:00<?, ? examples/s]

✅ SFTTrainer ready
   Base: CryptoYogi/vazhi-v7_0 (Gemma 3 1B-it, native Tamil)
   LR: 1e-05, Epochs: 1
   LoRA: r=16 (v7.0 was r=8), alpha=32, targets=['q_proj', 'v_proj']


In [10]:
# Cell 10 — Run Training

print(f"\U0001f680 Starting SFT v7.1 training (Gemma 3 — LoRA r={LORA_R})...")
print(f"   ~{total_steps} steps, {NUM_EPOCHS} epoch")
print(f"   Base: {BASE_MODEL}")
print(f"   LR: {LEARNING_RATE}, Dataset: {len(train_ds)} train")
print(f"   LoRA: r={LORA_R} (v7.0 was r=8)")
print()

train_result = trainer.train()

print("\n\u2705 Training complete!")
metrics = train_result.metrics
for k, v in metrics.items():
    print(f"   {k}: {v}")

print("\n\U0001f4ca Final eval...")
eval_metrics = trainer.evaluate()
for k, v in eval_metrics.items():
    print(f"   {k}: {v}")

if loss_cb.losses:
    s = loss_cb.losses[0][1]
    e = loss_cb.losses[-1][1]
    print(f"\n\U0001f4c8 Loss: {s:.4f} \u2192 {e:.4f} ({100*(s-e)/s:.1f}% drop)")

trainer.save_model()
trainer.push_to_hub()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


🚀 Starting SFT v7.1 training (Gemma 3 — LoRA r=16)...
   ~234 steps, 1 epoch
   Base: CryptoYogi/vazhi-v7_0
   LR: 1e-05, Dataset: 3754 train
   LoRA: r=16 (v7.0 was r=8)



Step,Training Loss,Validation Loss
78,4.228200,4.558148
156,4.178200,4.277536
234,4.181600,4.225536


  Step    7/234 | Loss: 4.8868 | LR: 2.50e-06
  Step   14/234 | Loss: 4.6375 | LR: 5.42e-06
  Step   21/234 | Loss: 4.8747 | LR: 8.33e-06
  Step   28/234 | Loss: 4.8019 | LR: 1.00e-05
  Step   35/234 | Loss: 4.8894 | LR: 9.94e-06
  Step   42/234 | Loss: 4.5692 | LR: 9.84e-06
  Step   49/234 | Loss: 4.4993 | LR: 9.68e-06
  Step   56/234 | Loss: 4.7284 | LR: 9.48e-06
  Step   63/234 | Loss: 4.4474 | LR: 9.22e-06
  Step   70/234 | Loss: 4.4829 | LR: 8.92e-06
  Step   77/234 | Loss: 4.2282 | LR: 8.57e-06
  📊 Eval Loss: 4.5581
  Step   84/234 | Loss: 4.3807 | LR: 8.19e-06
  Step   91/234 | Loss: 4.5371 | LR: 7.77e-06
  Step   98/234 | Loss: 4.4428 | LR: 7.33e-06
  Step  105/234 | Loss: 4.3139 | LR: 6.85e-06
  Step  112/234 | Loss: 4.1522 | LR: 6.36e-06

  🔍 Mid-training gen check (step 117):
    [greeting] Word: 100%, Rep: 0.00 | நான் உங்களுக்கு எப்படி உதவ முடியும்?
    [identity] Word: 92%, Rep: 0.00 | நான் கூகிள் பயிற்சி அளித்த ஒரு பெரிய மொழி மாதிரி. நான் Google ஆல் பயிற்றுவிக்கப்பட்டேன்.

  📊 Eval Loss: 4.2270
   eval_loss: 4.226979732513428
   eval_runtime: 10.2588
   eval_samples_per_second: 40.746
   eval_steps_per_second: 10.235

📈 Loss: 4.8868 → 4.1816 (14.4% drop)


...

  .../sft-v7_1/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...adapter_model.safetensors: 100%|##########| 5.98MB / 5.98MB            

  ...t/sft-v7_1/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/CryptoYogi/vazhi-v7_1-lora/commit/7cea4d46bbf4ffca2599321f1046b843ef98762e', commit_message='End of training', commit_description='', oid='7cea4d46bbf4ffca2599321f1046b843ef98762e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/CryptoYogi/vazhi-v7_1-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='CryptoYogi/vazhi-v7_1-lora'), pr_revision=None, pr_num=None)

In [11]:
# Cell 10r — Resume Training (if Colab disconnected)
# Uncomment and run ONLY if Cell 10 was interrupted

# print("Resuming training from checkpoint...")
# train_result = trainer.train(resume_from_checkpoint=True)
# trainer.save_model()
# trainer.push_to_hub()

print("Cell 10r: Resume cell (commented out). Uncomment only if needed.")

Cell 10r: Resume cell (commented out). Uncomment only if needed.


In [12]:
# Cell 11 — Save Adapter + A/B Test + Merge
#
# Test BOTH adapter inference and merged inference.
# If adapter works but merged doesn't, the merge is the problem.
# CRITICAL: Merge in fp16, NEVER into 4-bit (lesson from v3.6).

ADAPTER_PATH = f"{WORK_DIR}/vazhi-sft-v7_1-lora"

print("\U0001f4be Saving LoRA adapter...")
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"\u2705 Adapter saved to {ADAPTER_PATH}")

# Upload adapter backup
api = HfApi()
api.create_repo(ADAPTER_REPO, exist_ok=True)
print(f"\U0001f4e4 Uploading adapter to {ADAPTER_REPO}...")
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=ADAPTER_REPO,
    commit_message=f"SFT v7.1 adapter: Gemma 3 1B-it, {len(train_ds)} samples, r={LORA_R} (v7.0 was r=8), lr={LEARNING_RATE}",
)
print(f"\u2705 Adapter uploaded")

# Free training model
del model, trainer
gc.collect(); torch.cuda.empty_cache()
print("\U0001f5d1\ufe0f Training model freed")

AB_PROMPTS = [
    "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd",
    "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?",
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?",
    "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd",
    "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf",
]

# --- Test A: Adapter inference ---
print("\n" + "=" * 60)
print("\U0001f1e6 TEST A: Adapter Inference (no merge)")
print("=" * 60)

base_a = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0},
)
adapter_model = PeftModel.from_pretrained(base_a, ADAPTER_PATH)
adapter_model.eval()
adapter_model.config.use_cache = True

adapter_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(adapter_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    adapter_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

del adapter_model, base_a
gc.collect(); torch.cuda.empty_cache()

# --- Test B: Merged model ---
print("\n" + "=" * 60)
print("\U0001f1e7 TEST B: Merged Model (fp16)")
print("=" * 60)

base_b = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"":0},
)
peft_b = PeftModel.from_pretrained(base_b, ADAPTER_PATH)
peft_b.gradient_checkpointing_disable()
peft_b.config.use_cache = True
peft_b.eval()

print("\U0001f500 Merging LoRA in fp16...")
merged_model = peft_b.merge_and_unload()

merged_results = []
for prompt_text in AB_PROMPTS:
    resp = generate_response(merged_model, prompt_text)
    tw_pct, _, _ = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    merged_results.append({"resp": resp, "word": tw_pct, "rep": rep})
    print(f"  Q: {prompt_text}")
    print(f"  A: {resp[:200]}")
    print(f"  (Word:{tw_pct:.0f}% Rep:{rep:.2f})")
    print()

# A/B comparison
print("\n" + "=" * 60)
print("A/B COMPARISON (v7.1 — LoRA r=16)")
print("=" * 60)
avg_a_word = np.mean([r['word'] for r in adapter_results])
avg_b_word = np.mean([r['word'] for r in merged_results])
print(f"   Adapter avg word: {avg_a_word:.0f}%")
print(f"   Merged avg word:  {avg_b_word:.0f}%")
print(f"   v7.0 adapter was: 96%")
print(f"   v7.0 merged was:  98%")
if abs(avg_a_word - avg_b_word) > 15:
    print(f"   \u26a0\ufe0f MERGE CORRUPTION DETECTED \u2014 use adapter for deployment")
else:
    print(f"   \u2705 Merge OK \u2014 adapter and merged consistent")

💾 Saving LoRA adapter...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ft-v7_1/training_args.bin: 100%|##########| 6.29kB / 6.29kB            

  .../sft-v7_1/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

...

  ..._1-lora/training_args.bin: 100%|##########| 6.29kB / 6.29kB            

  ...v7_1-lora/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...adapter_model.safetensors: 100%|##########| 5.98MB / 5.98MB            

  ...-v7_1-lora/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

✅ Adapter uploaded
🗑️ Training model freed

🇦 TEST A: Adapter Inference (no merge)
  Q: வணக்கம்
  A: வணக்கம்! உங்களுக்கு என்ன உதவி வேண்டும்? நான் எப்படி உதவ முடியும்?
  (Word:100% Rep:0.00)

  Q: நீங்கள் யார்?
  A: நான் Google மூலம் பயிற்சி பெற்ற ஒரு பெரிய மொழி மாதிரி, நான் கூகிள் உருவாக்கியுள்ளேன்.
  (Word:92% Rep:0.00)

  Q: தமிழ்நாட்டின் தலைநகரம் எது?
  A: தமிழ்நாட்டின் தற்போதைய தலைநகரம் கோயம்புத்தூர்.
  (Word:100% Rep:0.00)

  Q: முதியோர் ஓய்வூதியம் பற்றி சொல்லுங்கள்
  A: அன்புள்ள பயணர், முதிர்ந்தவர் ஓய்வுப் பணம் பெற வேண்டும் என்று நினைத்தாலும் அது எப்போதும் இல்லை என்பது போல, உங்கள் வாழ்க்கையின் சில விஷயங்கள் விரைவாக மாறும் என்பதையும் கவனிக்கவும். 

முதிர்ந்தவர்கள் ஓய்
  (Word:100% Rep:0.00)

  Q: நன்றி
  A: எப்படி உதவ முடியும்?
  (Word:100% Rep:0.00)


🇧 TEST B: Merged Model (fp16)
🔀 Merging LoRA in fp16...
  Q: வணக்கம்
  A: வணக்கம்! நான் உங்களுக்கு எப்படி உதவ முடியும்?
  (Word:100% Rep:0.00)

  Q: நீங்கள் யார்?
  A: நான் கூகிள் மூலம் பயிற்சி பெற்ற ஒரு பெரிய மொழி மாதிரி.
  (Word:1

In [13]:
# Cell 12 — Full Eval: 16 Conversational Prompts
#
# Tamil WORD validation + human-readable output for quality assessment.
# Key question: Does SFT improve VAZHI identity/domain knowledge
# WITHOUT destroying Gemma 3's native Tamil capability?

merged_model.eval()
merged_model.config.use_cache = True

test_prompts = [
    {"prompt": "\u0bb5\u0ba3\u0b95\u0bcd\u0b95\u0bae\u0bcd", "check": "greeting", "cat": "greeting"},
    {"prompt": "\u0ba8\u0bc0\u0b99\u0bcd\u0b95\u0bb3\u0bcd \u0baf\u0bbe\u0bb0\u0bcd?", "check": "identity", "cat": "greeting"},
    {"prompt": "\u0b8e\u0ba9\u0b95\u0bcd\u0b95\u0bc1 \u0b89\u0ba4\u0bb5\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "general", "cat": "help"},
    {"prompt": "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "check": "general", "cat": "help"},
    {"prompt": "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bcd\u0b9f\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1?", "check": "factual", "cat": "factual"},
    {"prompt": "\u0b92\u0bb0\u0bc1 \u0ba4\u0bc6\u0bb0\u0bbf\u0baf\u0bbe\u0ba4 \u0b8e\u0ba3\u0bcd\u0ba3\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0bae\u0bc6\u0b9a\u0bc7\u0b9c\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0ba4\u0bc1. \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0bb5\u0ba4\u0bc1?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0bb5\u0bc0\u0b9f\u0bcd\u0b9f\u0bbf\u0bb2\u0bcd \u0ba4\u0bc0 \u0bb5\u0bbf\u0baa\u0ba4\u0bcd\u0ba4\u0bc1 \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "safety", "cat": "safety"},
    {"prompt": "\u0ba8\u0bbe\u0bb3\u0bc8 \u0baa\u0b99\u0bcd\u0b95\u0bc1 \u0b9a\u0ba8\u0bcd\u0ba4\u0bc8 \u0b8f\u0bb1\u0bc1\u0bae\u0bbe?", "check": "refusal", "cat": "refusal"},
    {"prompt": "\u0bae\u0bc1\u0ba4\u0bbf\u0baf\u0bcb\u0bb0\u0bcd \u0b93\u0baf\u0bcd\u0bb5\u0bc2\u0ba4\u0bbf\u0baf\u0bae\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "govt"},
    {"prompt": "\u0bb0\u0bc7\u0bb7\u0ba9\u0bcd \u0b95\u0bbe\u0bb0\u0bcd\u0b9f\u0bc1 \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd \u0ba4\u0bc7\u0bb5\u0bc8", "check": "domain", "cat": "govt"},
    {"prompt": "\u0ba8\u0bc0\u0bb0\u0bbf\u0bb4\u0bbf\u0bb5\u0bc1 \u0ba8\u0bcb\u0baf\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "health"},
    {"prompt": "\u0b95\u0bbe\u0baf\u0bcd\u0b9a\u0bcd\u0b9a\u0bb2\u0bcd \u0bb5\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf \u0bb5\u0bc7\u0ba3\u0bcd\u0b9f\u0bc1\u0bae\u0bcd?", "check": "domain", "cat": "health"},
    {"prompt": "\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1\u0bb1\u0bb3\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd", "check": "domain", "cat": "culture"},
    {"prompt": "FIR \u0baa\u0bcb\u0b9f\u0bc1\u0bb5\u0ba4\u0bc1 \u0b8e\u0baa\u0bcd\u0baa\u0b9f\u0bbf?", "check": "domain", "cat": "legal"},
    {"prompt": "\u0b95\u0bb2\u0bcd\u0bb5\u0bbf \u0b95\u0b9f\u0ba9\u0bcd \u0baa\u0bb1\u0bcd\u0bb1\u0bbf \u0ba4\u0b95\u0bb5\u0bb2\u0bcd", "check": "domain", "cat": "education"},
    {"prompt": "\u0b9a\u0bc8\u0baa\u0bb0\u0bcd \u0bae\u0bcb\u0b9a\u0b9f\u0bbf\u0baf\u0bbf\u0bb2\u0bcd \u0b87\u0bb0\u0bc1\u0ba8\u0bcd\u0ba4\u0bc1 \u0baa\u0ba3\u0bae\u0bcd \u0b87\u0bb4\u0ba8\u0bcd\u0ba4\u0bbe\u0bb2\u0bcd \u0b8e\u0ba9\u0bcd\u0ba9 \u0b9a\u0bc6\u0baf\u0bcd\u0baf\u0bb2\u0bbe\u0bae\u0bcd?", "check": "safety", "cat": "security"},
]

print(f"{'='*70}")
print(f"\U0001f4ca FULL EVAL: 16 Conversational Prompts")
print(f"{'='*70}")

results = []
for item in test_prompts:
    resp = generate_response(merged_model, item['prompt'])
    t_pct = tamil_char_pct(resp)
    tw_pct, tw_count, tw_total = tamil_word_score(resp)
    rep = compute_repeat_ratio(resp)
    is_empty = len(resp.strip()) < 10

    results.append({
        'prompt': item['prompt'], 'cat': item['cat'],
        'response': resp[:300], 'tamil_char_pct': t_pct,
        'tamil_word_pct': tw_pct, 'repeat_ratio': rep,
        'is_empty': is_empty,
    })

    status = "\u274c EMPTY" if is_empty else ("\u26a0\ufe0f REP" if rep > 0.3 else "\u2705")
    print(f"\n[{item['cat']:>10}] {status} | Char: {t_pct:.0f}%, Word: {tw_pct:.0f}%, Rep: {rep:.2f}")
    print(f"  Q: {item['prompt']}")
    print(f"  A: {resp[:200]}")

# Summary
avg_char = np.mean([r['tamil_char_pct'] for r in results])
avg_word = np.mean([r['tamil_word_pct'] for r in results])
avg_rep = np.mean([r['repeat_ratio'] for r in results])
non_empty = sum(1 for r in results if not r['is_empty'])
high_rep = sum(1 for r in results if r['repeat_ratio'] > 0.3)

print(f"\n{'='*70}")
print(f"\U0001f4ca EVAL SUMMARY")
print(f"{'='*70}")
print(f"   Non-empty:      {non_empty}/{len(results)}")
print(f"   Avg Tamil char: {avg_char:.0f}%")
print(f"   Avg Tamil word: {avg_word:.0f}%")
print(f"   Avg repeat:     {avg_rep:.2f}")
print(f"   High repeat:    {high_rep}/{len(results)}")
print(f"")
print(f"   Vanilla baseline: char {avg_pre_char:.0f}%, word {avg_pre_word:.0f}%")
print(f"   Post-SFT:         char {avg_char:.0f}%, word {avg_word:.0f}%")
print(f"   \u0394 Char:           {avg_char - avg_pre_char:+.0f}%")
print(f"   \u0394 Word:           {avg_word - avg_pre_word:+.0f}%")

# GO / NO-GO
# For Gemma 3, the bar is HIGHER: we already start with real Tamil.
# SFT must NOT degrade Tamil quality (Navarasa failure mode).
TAMIL_NOT_DEGRADED = (avg_word >= avg_pre_word - 10)  # Allow max 10% drop
EVAL_PASSED = (non_empty >= 12 and avg_word >= 50 and high_rep <= 3 and TAMIL_NOT_DEGRADED)

if EVAL_PASSED:
    print(f"\n   \u2705 EVAL PASSED \u2014 proceed to upload")
else:
    print(f"\n   \u274c EVAL FAILED")
    if non_empty < 12:
        print(f"     Too many empty responses ({len(results) - non_empty})")
    if avg_word < 50:
        print(f"     Tamil word % too low ({avg_word:.0f}%)")
    if high_rep > 3:
        print(f"     Too many repetitive outputs ({high_rep})")
    if not TAMIL_NOT_DEGRADED:
        print(f"     \u26a0\ufe0f CATASTROPHIC FORGETTING: Tamil word dropped from {avg_pre_word:.0f}% to {avg_word:.0f}%")
        print(f"     Navarasa-style failure \u2014 try lower LR or fewer LoRA modules")

📊 FULL EVAL: 16 Conversational Prompts

[  greeting] ✅ | Char: 95%, Word: 100%, Rep: 0.00
  Q: வணக்கம்
  A: சரியாக! நான் உங்களுக்கு எப்படி உதவ முடியும்?

[  greeting] ✅ | Char: 98%, Word: 100%, Rep: 0.00
  Q: நீங்கள் யார்?
  A: நான் ஒரு பெரிய மொழி மாதிரி, கூகிளால் பயிற்சி அளிக்கப்பட்டது. நான் பல்வேறு வகையான பணிகளைச் செய்யக்கூடிய திறனைக் கொண்டிருக்கிறேன், இதில் உரையை உருவாக்குதல், மொழியைத் தருநாகம் மற்றும் கேள்விகளுக்குப் பதி

[      help] ✅ | Char: 94%, Word: 100%, Rep: 0.00
  Q: எனக்கு உதவுங்கள்
  A: சரி, நான் உங்களுக்கு எப்படி உதவ முடியும்?

[      help] ✅ | Char: 95%, Word: 100%, Rep: 0.00
  Q: நன்றி
  A: சரியாக! உங்களுக்கு நான் எப்படி உதவ முடியும்?

[   factual] ✅ | Char: 92%, Word: 95%, Rep: 0.00
  Q: தமிழ்நாட்டின் தலைநகரம் எது?
  A: தமிழ்நாட்டின் столица என்பது கொல்கத்தா ஆகும். இது ஒரு பெரிய நகரம் மற்றும் முக்கிய வணிக மையமாகும், மேலும் அது பல கலாச்சாரங்களின் தாயகமாகவும் உள்ளது.

[    safety] ✅ | Char: 97%, Word: 96%, Rep: 0.00
  Q: ஒரு தெரியாத எண்ணில் இருந்து மெசேஜ் வந்தது. என்ன 

In [14]:
# Cell 13 — Upload Merged Model

if not EVAL_PASSED:
    print("\u274c Eval did not pass. Skipping upload.")
    print(f"   Adapter available at: {ADAPTER_REPO}")
    print(f"")
    print(f"   Possible fixes:")
    print(f"   1. Lower LR to 5e-6 (current: {LEARNING_RATE})")
    print(f"   2. Reduce LoRA rank back to r=8 (current: r={LORA_R})")
    print(f"   3. Try r=16 with fewer target modules")
    print(f"   4. Check if r=16 caused catastrophic forgetting")
else:
    MERGED_PATH = f"{WORK_DIR}/vazhi-v7_1-merged"

    print(f"\U0001f4be Saving merged model to {MERGED_PATH}...")
    merged_model.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)

    api = HfApi()
    api.create_repo(OUTPUT_MODEL, exist_ok=True)

    print(f"\U0001f4e4 Uploading merged model to {OUTPUT_MODEL}...")
    api.upload_folder(
        folder_path=MERGED_PATH,
        repo_id=OUTPUT_MODEL,
        commit_message=(
            f"SFT v7.1: VAZHI Tamil assistant | "
            f"base={BASE_MODEL} (Gemma 3 1B-it) | "
            f"LoRA r={LORA_R} (v7.0 was r=8) x {LORA_TARGETS} | "
            f"lr={LEARNING_RATE} | {len(train_ds)} samples | "
            f"Tamil word: {avg_pre_word:.0f}% -> {avg_word:.0f}%"
        ),
    )

    print(f"\n\u2705 Merged model: https://huggingface.co/{OUTPUT_MODEL}")
    print(f"\u2705 Adapter:      https://huggingface.co/{ADAPTER_REPO}")

💾 Saving merged model to /content/vazhi-v7_1-merged...
📤 Uploading merged model to CryptoYogi/vazhi-v7_1...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._1-merged/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...7_1-merged/tokenizer.json: 100%|##########| 33.4MB / 33.4MB            

  ...-merged/model.safetensors:   2%|1         | 33.5MB / 2.00GB            


✅ Merged model: https://huggingface.co/CryptoYogi/vazhi-v7_1
✅ Adapter:      https://huggingface.co/CryptoYogi/vazhi-v7_1-lora


In [15]:
# Cell 14 — Summary

print(f"{'='*65}")
print(f"\U0001f4cb SFT v7.1 TRAINING SUMMARY (Incremental + LoRA r=16)")
print(f"{'='*65}")
print(f"")
print(f"   Lineage:     google/gemma-3-1b-it \u2192 v7.0 (r=8) \u2192 v7.1 (r=16)")
print(f"   Base model:  {BASE_MODEL}")
print(f"   Dataset:     {SFT_DATASET} ({len(train_ds)} train / {len(eval_ds)} eval)")
print(f"   Output:      {OUTPUT_MODEL}")
print(f"")
print(f"   Training:")
print(f"     LR:          {LEARNING_RATE}")
print(f"     LoRA:        r={LORA_R} (v7.0 was r=8), alpha={LORA_ALPHA}, targets={LORA_TARGETS}")
print(f"     Epochs:      {NUM_EPOCHS} (total: 2 counting v7.0)")
print(f"     Steps:       ~{total_steps}")
print(f"")
print(f"   Results:")
print(f"     Tamil char:  {avg_pre_char:.0f}% \u2192 {avg_char:.0f}% (\u0394 {avg_char - avg_pre_char:+.0f}%)")
print(f"     Tamil word:  {avg_pre_word:.0f}% \u2192 {avg_word:.0f}% (\u0394 {avg_word - avg_pre_word:+.0f}%)")
print(f"     Non-empty:   {non_empty}/{len(results)}")
print(f"     Avg repeat:  {avg_rep:.2f}")
print(f"     Eval passed: {'\u2705 YES' if EVAL_PASSED else '\u274c NO'}")
print(f"")
print(f"   Progression (vanilla \u2192 v7.0 \u2192 v7.1):")
print(f"     Vanilla:     char 93%, word 95% — no VAZHI identity")
print(f"     v7.0 (r=8):  char 92%, word 94% — partial identity, factual wrong")
print(f"     v7.1 (r=16): char {avg_char:.0f}%, word {avg_word:.0f}%")
print(f"")
print(f"   \U0001f449 Next steps:")
if EVAL_PASSED:
    print(f"      1. GGUF conversion (Q4_K_M = ~0.60 GB for mobile)")
    print(f"      2. Test GGUF output quality (Tamil coherence)")
    print(f"      3. Integrate with Flutter app")
    print(f"      4. Test on mobile device")
    if avg_word < avg_pre_word - 5:
        print(f"      \u26a0\ufe0f  Tamil dropped {avg_pre_word - avg_word:.0f}% — consider r=12 middle ground")
else:
    print(f"      1. r=16 incremental was too aggressive — Tamil quality degraded")
    print(f"      2. Try r=8 with epoch 2 from vanilla (simpler, less aggressive)")
    print(f"      3. Or try adding more target modules instead of higher rank")

📋 SFT v7.1 TRAINING SUMMARY (Incremental + LoRA r=16)

   Lineage:     google/gemma-3-1b-it → v7.0 (r=8) → v7.1 (r=16)
   Base model:  CryptoYogi/vazhi-v7_0
   Dataset:     CryptoYogi/vazhi-tamil-sft-v7_0 (3754 train / 418 eval)
   Output:      CryptoYogi/vazhi-v7_1

   Training:
     LR:          1e-05
     LoRA:        r=16 (v7.0 was r=8), alpha=32, targets=['q_proj', 'v_proj']
     Epochs:      1 (total: 2 counting v7.0)
     Steps:       ~234

   Results:
     Tamil char:  89% → 95% (Δ +7%)
     Tamil word:  90% → 96% (Δ +6%)
     Non-empty:   16/16
     Avg repeat:  0.00
     Eval passed: ✅ YES

   Progression (vanilla → v7.0 → v7.1):
     Vanilla:     char 93%, word 95% — no VAZHI identity
     v7.0 (r=8):  char 92%, word 94% — partial identity, factual wrong
     v7.1 (r=16): char 95%, word 96%

   👉 Next steps:
      1. GGUF conversion (Q4_K_M = ~0.60 GB for mobile)
      2. Test GGUF output quality (Tamil coherence)
      3. Integrate with Flutter app
      4. Test on mobile d